**URL Anomaly Pattern-Family Discovery for Feature Engineering**

Analyse the characteristics of **anomalous-only** URLs to derive URL-shape patterns for the featurisation stage. URLs are analysed in **iterative batches**, each batch's observed anomaly shapes are documented and categorised into **pattern families**. The families are verified on two axes — coverage in the anomalous-only URL set and precision i.e no family pattern matches any normal URL set before being frozen as the basis for URL feature definitions for model training

In [12]:
import os
import re
import random
from pathlib import Path
import json
import pandas as pd

In [13]:
# working directory for relative paths
notebooks_path = Path.cwd() # <repo-root>/noteboooks
repo_root_path = notebooks_path.parent # <repo-root>/
data_path = repo_root_path / "data" # <repo-root>/data
anom_only_urls_file = repo_root_path / "data" / "anom_only_urls.paraquet"
norm_only_urls_file = repo_root_path / "data" / "norm_only_urls.paraquet"

**Load the anomaly and normal URLs dataset**

In [14]:
print("Anomaly URLs dataset:")
anom_only_urls_df = pd.read_parquet(anom_only_urls_file)
print(f"loaded: {anom_only_urls_file}")
print(f"shape: {anom_only_urls_df.shape}")
print("")

print("Normal URLs dataset:")
norm_only_urls_df = pd.read_parquet(norm_only_urls_file)
print(f"loaded: {norm_only_urls_file}")
print(f"shape: {norm_only_urls_df.shape}")

Anomaly URLs dataset:
loaded: /mnt/hgfs/workspace/mlhttp/data/anom_only_urls.paraquet
shape: (1615, 2)

Normal URLs dataset:
loaded: /mnt/hgfs/workspace/mlhttp/data/norm_only_urls.paraquet
shape: (20, 2)


Anomalous URLs dataset contains `1615` and Normal URLs dataset contains `20` unique URLs

**Method - iterative pattern-family discovery**

- Inspect the highest-frequency unmatched URLs in the remaining anomalous-only set
- Mine the observed anomaly shapes into a named pattern family - patterns are derived from observed URLs
- Recompile and re-peel the full anomalous set - every run restarts from the complete dataset, so results are deterministic and state-independent: no run depends on the output of a previous run
- Repeat until coverage converges: every anomalous URL is matched by some family (1,615 URLs / 5,508 requests, remaining = 0)
- Precision check - run each family pattern against the normal-only URL set. Violations are tightened
- Rehoming - any anomalous coverage lost by tightening is reconciled by an existing or new pattern family. Re-check the full partition after every change
- Freeze the pattern families - complete coverage of the anomalous set, zero hits on the normal set

**Pattern-Family Utilities**

Helper functions that manage the pattern-family registry and apply compiled patterns to URL datasets. Two complementary scanners methods:
- `peel_matched_urls` - This is to check `coverage` on anomalous set. Sequentially removes family matches one by one, returns the remainder unmatched set.
- `collect_matched_urls` - This is to check the `precision` of patterns on normal set. Every family is evaluated against the full dataset. The matched subsets are returned per family.

In [15]:
# Pattern-family registry
pattern_families = {}

def _require(family: str):
    if family not in pattern_families:
        raise KeyError(f"family '{family}' not created — call new_pattern_family first")

def new_pattern_family(family: str):
    """Create an empty pattern family (list of regex alternatives)."""
    pattern_families[family] = []

def extend_pattern(family: str, *patterns: str):
    """Add regex alternatives to a family (duplicates ignored)."""
    _require(family)
    pattern_families[family].extend(p for p in patterns if p not in pattern_families[family])

def overwrite_pattern(family: str, *patterns: str):
    """Replace a family's patterns entirely (used by the precision pass)."""
    _require(family)
    pattern_families[family] = list(patterns)

def compiled_pattern(families: dict) -> dict:
    """Join each family's alternatives into one regex string."""
    return {name: "|".join(dict.fromkeys(pats)) for name, pats in families.items()}

def peel_matched_urls(df_input, compiled, count_col="anom"):
    """Sequential disjoint peel (anom side): each family claims matches
    from the remaining set in registry order; returns the unmatched remainder."""
    df_remaining = df_input.copy()
    for name, pat in compiled.items():
        urls = df_remaining.index.to_series()
        mask = urls.str.contains(pat, case=False, regex=True, na=False).values

        df_suffix   = df_remaining.loc[mask].copy()
        df_remaining = df_remaining.loc[~mask].copy()

        print(f"{name}: {pat}")
        print(f"{name} URLs: {len(df_suffix)} URLs, {df_suffix[count_col].sum()} requests")
        print(f"remaining:  {len(df_remaining)} URLs, {df_remaining[count_col].sum()} requests")
        print("-" * 80)
    return df_remaining

def collect_matched_urls(df_input, compiled):
    """Independent scan (norm side): every family evaluated against the full
    dataset; returns {family: matched_df} for families with >= 1 match ({} = clean)."""
    urls = df_input.index.to_series()
    matched = {}
    for name, pat in compiled.items():
        mask = urls.str.contains(pat, case=False, regex=True, na=False).values
        if mask.any():
            matched[name] = df_input.loc[mask].copy()
            print(f"'{name}' family matched URLs: {len(matched[name])}")
    print(f"*** {'CLEAN: no family pattern matches any URL' if not matched else f'{len(matched)} families fire'} ***")
    return matched

**Iteration 1:** Backup-Extension and Numeric-ID Families

Inspect the first batch of URLs

In [16]:
anom_only_urls_df.head(30)

label,anom,norm
url,,
http://localhost:8080.BAK,19,0
http://localhost:8080.Bak,20,0
http://localhost:8080.INC,13,0
http://localhost:8080.Inc,25,0
http://localhost:8080.OLD,20,0
http://localhost:8080.Old,22,0
http://localhost:8080.bak,23,0
http://localhost:8080.inc,16,0
http://localhost:8080.java,16,0


**Observations:**  

Multiple Malformed short URLs variations, all mutating the host root rather than any application path:  
- Backup-file extensions in three case variants `.BAK, .Bak, .bak` `.INC`, `.Inc`, `.inc` `.OLD`, `.Old`, `.old`, `java`, `.jsp`
- Placement variants - `http://localhost:8080` followed by,
    -  (**No** `/`) direct `.` + extension
    -  `/` only - host + trailing slash
    -  `/` + `.` + extension - dot-file form
    - `/` + large numeric value (~19 digits)
    - `/` + large numeric value + extension

Group these into two pattern families - 
- placement variants 1-3: `extensions` pattern family group
- numeric variants 4-5: `numeric-id` pattern family group

*The `.jsp` pattern is included provisionally, it is revisited during the normal URLs dataset violation pass*  
*The **host + trailing slash** pattern is revisited in the subsequent URL batches as more variants of URLs with trailing slash are discovered*  
*The `numeric-id` pattern group has overlap with `extensions` pattern group due to registry order*

Create the pattern families, compile, and peel from the full dataset

In [17]:
# Patterns mined from the observed URLs
extension_pattern = r"\.(?:bak|old|inc|java|jsp)"
numeric_id_pattern = r"\d{10,}"

new_pattern_family("extensions")
extend_pattern("extensions", extension_pattern)

new_pattern_family("numeric-id")
extend_pattern("numeric-id", numeric_id_pattern)

compiled = compiled_pattern(pattern_families)

# Reset to the full anomalous dataset — every iteration re-peels from scratch
anom_only_remaining_df = anom_only_urls_df.copy()
print(anom_only_remaining_df.shape)

anom_only_remaining_df = peel_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions URLs: 1308 URLs, 3398 requests
remaining:  307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id URLs: 26 URLs, 174 requests
remaining:  281 URLs, 1936 requests
--------------------------------------------------------------------------------


The first two families `extensions` and `numeric-id` cover `1308` and `26` URLs respectively i.e a total of 1,334 of 1,615 anomalous URLs

Verify partition and registry after first iteration

Lock the running partition numbers for reproducibility

In [36]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

# Set the actual result
actual_result = {
    "extensions": { 
        "matched": { "urls": 1308, "requests": 3398 },
        "remaining": { "urls": 307, "requests": 2110 },
    },
    "numeric_id": {
        "matched": { "urls": 26, "requests": 174},
        "remaining": { "urls": 281, "requests": 1936},
    }
}

# Locked expected result
expected_result = {
    "extensions": { 
        "matched": { "urls": 1308, "requests": 3398 },
        "remaining": { "urls": 307, "requests": 2110 },
    },
    "numeric_id": {
        "matched": { "urls": 26, "requests": 174},
        "remaining": { "urls": 281, "requests": 1936},
    }
}

def verify_results(actual_result, expected_result):
    actual_matched_urls = 0
    actual_matched_requests = 0
    for key, value in expected_result.items():
        # Verify regex expression hits on URLs and requests 
        assert actual_result[key]["matched"]["urls"] == expected_result[key]["matched"]["urls"]
        assert actual_result[key]["matched"]["requests"] == expected_result[key]["matched"]["requests"]
        # Verify remaining URLs and requests in dataframe after peel 
        assert actual_result[key]["remaining"]["urls"] == expected_result[key]["remaining"]["urls"]
        assert actual_result[key]["remaining"]["requests"] == expected_result[key]["remaining"]["requests"]
        
        actual_matched_urls += actual_result[key]["matched"]["urls"]
        actual_matched_requests += actual_result[key]["matched"]["requests"]

    # Running patterns hits + count in peel'ed dataframe = Total
    assert (actual_matched_urls + len(anom_only_remaining_df))  == len(anom_only_urls_df)
    assert (actual_matched_requests + anom_only_remaining_df["anom"].sum())  == anom_only_urls_df["anom"].sum()

verify_results(actual_result, expected_result)

Dataframe shape: (281, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ]
}


The peel'ed (remaining) dataframe contains `281` URLs  
The registry holds two verified families (`extensions`, `numeric-id`)  

**Iteration 2:**  Product-Scanner Probes (ColdFusion, WebSphere, IIS) + Root Trailing Slash

Inspect the second batch of URLs

In [37]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/,51,0
http://localhost:8080/CFIDE/administrator/Application.cfm,5,0
http://localhost:8080/CFIDE/administrator/aboutcf.cfm,7,0
http://localhost:8080/CFIDE/administrator/checkfile.cfm,6,0
http://localhost:8080/CFIDE/administrator/enter.cfm,9,0
http://localhost:8080/CFIDE/administrator/header.cfm,9,0
http://localhost:8080/CFIDE/administrator/homefile.cfm,4,0
http://localhost:8080/CFIDE/administrator/homepage.cfm,8,0
http://localhost:8080/CFIDE/administrator/index.cfm,9,0


**Observations:**  

Multiple Malformed URLs variations:  
- Trailing slash `http://localhost:8080/` 	
- Adobe coldfusion scan probes - URLs with `CFIDE/...`
- IBM WebSphere scan probes - URLs with (`IBMWebAS`)
- MS IIS server scan probes - URLs with (`IISSamples/...`)

Group each one into its own respective pattern family 

In [38]:
# Patterns mined from the observed URLs
trailing_slash_pattern = (r"^https?://[^/:]+:\d{1,5}/$")
adobe_coldfusion_patterns = (r"CFIDE/")
websphere_patterns = (r"IBMWebAS")
iis_patterns = (r"IISSamples")

new_pattern_family("trailing_slash")
extend_pattern("trailing_slash", trailing_slash_pattern)

new_pattern_family("adobe_coldfusion")
extend_pattern("adobe_coldfusion", adobe_coldfusion_patterns)

new_pattern_family("websphere")
extend_pattern("websphere", websphere_patterns)

new_pattern_family("iis")
extend_pattern("iis", iis_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to the full anomalous dataset — every iteration re-peels from scratch
anom_only_remaining_df = anom_only_urls_df.copy()
print(anom_only_remaining_df.shape)

anom_only_remaining_df = peel_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions URLs: 1308 URLs, 3398 requests
remaining:  307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id URLs: 26 URLs, 174 requests
remaining:  281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash URLs: 1 URLs, 51 requests
remaining:  280 URLs, 1885 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion URLs: 20 URLs, 136 requests
remaining:  260 URLs, 1749 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS
websphere URLs: 1 URLs, 11 requests
remaining:  259 URLs, 1738 requests
--------------------------------------------------------------------------------
iis: IISSamples
iis URLs: 21 URLs, 157 requests
remaini

Pattern families coverage:  
- `extensions` 1308 URLs
- `numeric-id` 26 URLs
- `trailing_slash` 1 URL
- `adobe_coldfusion` 20 URLs
- `websphere` 1 URL
- `iis` 21 URLs

The batch two families `trailing_slash`, `adobe_coldfusion` `websphere` and `iis` increased the coverage by `43` URLs on top of previous batch families  i.e a total of `1,377` of `1,615` anomalous URLs

Verify partition and registry after second iteration

In [41]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

# Set the actual results
actual_result = {
    "extensions": { 
        "matched": { "urls": 1308, "requests": 3398 },
        "remaining": { "urls": 307, "requests": 2110 },
    },
    "numeric_id": {
        "matched": { "urls": 26, "requests": 174},
        "remaining": { "urls": 281, "requests": 1936},
    },
    "trailing_slash": { 
        "matched": { "urls": 1, "requests": 51 },
        "remaining": { "urls": 280, "requests": 1885 },
    },
    "adobe_coldfusion": {
        "matched": { "urls": 20, "requests": 136},
        "remaining": { "urls": 260, "requests": 1749},
    },
    "websphere": {
        "matched": { "urls": 1, "requests": 11},
        "remaining": { "urls": 259, "requests": 1738},
    },
    "iis": {
        "matched": { "urls": 21, "requests": 157},
        "remaining": { "urls": 238, "requests": 1581},
    }
}

# Locked expected result
# No channge in the `extensions` and `numeric_id` values from previous iteration
expected_result.update({
    "trailing_slash": { 
        "matched": { "urls": 1, "requests": 51 },
        "remaining": { "urls": 280, "requests": 1885 },
    },
    "adobe_coldfusion": {
        "matched": { "urls": 20, "requests": 136},
        "remaining": { "urls": 260, "requests": 1749},
    },
    "websphere": {
        "matched": { "urls": 1, "requests": 11},
        "remaining": { "urls": 259, "requests": 1738},
    },
    "iis": {
        "matched": { "urls": 21, "requests": 157},
        "remaining": { "urls": 238, "requests": 1581},
    }
})

verify_results(actual_result, expected_result)

Dataframe shape: (238, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS"
  ],
  "iis": [
    "IISSamples"
  ]
}


The peel'ed (remaining) dataframe contains `238` URLs  
The registry holds six verified families `extensions` `numeric-id` `trailing_slash` `adobe_coldfusion` `websphere` and `iis` 

Iteration 3: TBD

Inspect the third batch of URLs

In [43]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/SmarTicketApp/index.html,6,0
http://localhost:8080/WEB-INF/web.xml,4,0
http://localhost:8080/WEB-INF/webapp.properties,3,0
http://localhost:8080/WebSphereSamples/,23,0
http://localhost:8080/WebSphereSamples/SingleSamples/AccountAndTransfer/create.html,8,0
http://localhost:8080/WebSphereSamples/SingleSamples/Increment/increment.html,11,0
http://localhost:8080/WebSphereSamples/YourCo/main.html,5,0
http://localhost:8080/_cti_pvt/access.cnf,3,0
http://localhost:8080/_cti_pvt/botinfs.cnf,8,0


**Observations:**  

Multiple Malformed URLs variations:  
- More variations of IBM WebSphere scan probes - URLs with (`webapp...`, `WebSphere...`, `SmarTicketApp...`)
- Microsoft frontpage scan probes - URLs with (`_cti...`, `_private...`, `_vti...`)
- Java metadata probes - URLs with `WEB-INF`

Extend the websphere pattern family and make new one for ms frontpage and Java app

In [46]:
# Patterns mined from the observed URLs
more_websphere_patterns = (r"webapp|WebSphere|SmarTicketApp")
frontpage_patterns = (r"_cti|_private|_vti")
java_patterns = (r"WEB-INF")

extend_pattern("websphere", more_websphere_patterns)

new_pattern_family("frontpage")
extend_pattern("frontpage", frontpage_patterns)

new_pattern_family("java")
extend_pattern("java", java_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to the full anomalous dataset — every iteration re-peels from scratch
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = peel_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions URLs: 1308 URLs, 3398 requests
remaining:  307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id URLs: 26 URLs, 174 requests
remaining:  281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash URLs: 1 URLs, 51 requests
remaining:  280 URLs, 1885 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion URLs: 20 URLs, 136 requests
remaining:  260 URLs, 1749 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp
websphere URLs: 15 URLs, 136 requests
remaining:  245 URLs, 1613 requests
--------------------------------------------------------------------------------
iis: IISSamples
iis UR

In [47]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

# Set the actual results
actual_result = {
    "extensions": { 
        "matched": { "urls": 1308, "requests": 3398 },
        "remaining": { "urls": 307, "requests": 2110 },
    },
    "numeric_id": {
        "matched": { "urls": 26, "requests": 174},
        "remaining": { "urls": 281, "requests": 1936},
    },
    "trailing_slash": { 
        "matched": { "urls": 1, "requests": 51 },
        "remaining": { "urls": 280, "requests": 1885 },
    },
    "adobe_coldfusion": {
        "matched": { "urls": 20, "requests": 136},
        "remaining": { "urls": 260, "requests": 1749},
    },
    "websphere": {
        "matched": { "urls": 15, "requests": 136},
        "remaining": { "urls": 245, "requests": 1613},
    },
    "iis": {
        "matched": { "urls": 21, "requests": 157},
        "remaining": { "urls": 224, "requests": 1456},
    },
    "frontpage": {
        "matched": { "urls": 23, "requests": 181},
        "remaining": { "urls": 201, "requests": 1275},
    },
    "java": {
        "matched": { "urls": 1, "requests": 4},
        "remaining": { "urls": 200, "requests": 1271},
    }
}

# Locked expected result
# No channge in the `extensions` and `numeric_id` values from previous iteration
expected_result.update({
    # Updated websphere and iis numbers due to results of reworked patterns in this iteration
    "websphere": {
        "matched": { "urls": 15, "requests": 136},
        "remaining": { "urls": 245, "requests": 1613},
    },
    "iis": {
        "matched": { "urls": 21, "requests": 157},
        "remaining": { "urls": 224, "requests": 1456},
    },

    # new families in this iteration
    "frontpage": {
        "matched": { "urls": 23, "requests": 181},
        "remaining": { "urls": 201, "requests": 1275},
    },
    "java": {
        "matched": { "urls": 1, "requests": 4},
        "remaining": { "urls": 200, "requests": 1271},
    }    
})

verify_results(actual_result, expected_result)

Dataframe shape: (200, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp"
  ],
  "iis": [
    "IISSamples"
  ],
  "frontpage": [
    "_cti|_private|_vti"
  ],
  "java": [
    "WEB-INF"
  ]
}


Iteration 4: TBD

Inspect the fourth batch of URLs

In [50]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/iisstart.asp,4,0
http://localhost:8080/iisstart.htm,9,0
http://localhost:8080/jstl-war/index.html,9,0
http://localhost:8080/mail.box,6,0
http://localhost:8080/msadc/msadcs.dll,16,0
http://localhost:8080/msadc/readme.txt,2,0
http://localhost:8080/msadc/samples/adctest.asp,6,0
http://localhost:8080/msadc/samples/selector/showcode.asp,8,0
http://localhost:8080/postinfo.html,5,0


**Observations:**  

Multiple Malformed URLs variations: 
- Lotus Domino databases probes - - URLs with (`.nsf`)
- Apache Software Foundation probes - - URLs with (`asf-logo...`)
- More variations of IIS server scan probes - URLs with (`iisadmin...`, `iisadmpwd`)
- More variations of IBM WebSphere/J2EE scan probes - URLs with (`estore...`)
- WebLogic Portal probes - URLs with (`e2ePortalProject...`)
- Apache Struts application probe - URLs with (`admin/login.do`)
- Allaire/Macromedia JRun application probe - URLs with (`flash/java/javabean/FlashJavaBean...`)
- Known scanner wordlist probe - URLs with `databasenotes...`

In [51]:
# Patterns mined from the observed URLs
lotus_domino_patterns = (r"\.nsf$")
asf_patterns = (r"asf-logo-wide")
more_iis_patterns = (r"iisadmin|iisadmpwd")
more_websphere_patterns = (r"estore")
portal_app_pattern = (r"e2ePortalProject")
more_java_patterns = (r"\.do$|flash/java/javabean|FlashJavaBean")
file_patterns = (r"databasenotes\.html")

new_pattern_family("lotus_domino")
extend_pattern("lotus_domino", lotus_domino_patterns)

new_pattern_family("asf")
extend_pattern("asf", asf_patterns)

extend_pattern("iis", more_iis_patterns)

extend_pattern("websphere", more_websphere_patterns)

new_pattern_family("portal_app")
extend_pattern("portal_app", portal_app_pattern)

extend_pattern("java", more_java_patterns)

new_pattern_family("file")
extend_pattern("file", file_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to the full anomalous dataset — every iteration re-peels from scratch
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = peel_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions URLs: 1308 URLs, 3398 requests
remaining:  307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id URLs: 26 URLs, 174 requests
remaining:  281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash URLs: 1 URLs, 51 requests
remaining:  280 URLs, 1885 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion URLs: 20 URLs, 136 requests
remaining:  260 URLs, 1749 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore
websphere URLs: 18 URLs, 149 requests
remaining:  242 URLs, 1600 requests
--------------------------------------------------------------------------------
iis: IISSamples

In [53]:
# Set the actual results
actual_result = {
    "extensions": { 
        "matched": { "urls": 1308, "requests": 3398 },
        "remaining": { "urls": 307, "requests": 2110 },
    },
    "numeric_id": {
        "matched": { "urls": 26, "requests": 174},
        "remaining": { "urls": 281, "requests": 1936},
    },
    "trailing_slash": { 
        "matched": { "urls": 1, "requests": 51 },
        "remaining": { "urls": 280, "requests": 1885 },
    },
    "adobe_coldfusion": {
        "matched": { "urls": 20, "requests": 136},
        "remaining": { "urls": 260, "requests": 1749},
    },
    "websphere": {
        "matched": { "urls": 18, "requests": 149},
        "remaining": { "urls": 242, "requests": 1600},
    },
    "iis": {
        "matched": { "urls": 30, "requests": 222},
        "remaining": { "urls": 212, "requests": 1378},
    },
    "frontpage": {
        "matched": { "urls": 23, "requests": 181},
        "remaining": { "urls": 189, "requests": 1197},
    },
    "java": {
        "matched": { "urls": 6, "requests": 41},
        "remaining": { "urls": 183, "requests": 1156},
    },
    "lotus_domino": {
        "matched": { "urls": 18, "requests": 128},
        "remaining": { "urls": 165, "requests": 1028},
    },
    "asf": {
        "matched": { "urls": 72, "requests": 529},
        "remaining": { "urls": 93, "requests": 499},
    },
    "portal_app": {
        "matched": { "urls": 1, "requests": 4},
        "remaining": { "urls": 92, "requests": 495},
    },
    "file": {
        "matched": { "urls": 1, "requests": 4},
        "remaining": { "urls": 91, "requests": 491},
    }
}

# Locked expected result
# No channge in the `extensions` and `numeric_id` values from previous iteration
expected_result.update({
    # Updated websphere, iis, frontpage and java, numbers due to results of reworked patterns in this iteration
    "websphere": {
        "matched": { "urls": 18, "requests": 149},
        "remaining": { "urls": 242, "requests": 1600},
    },
    "iis": {
        "matched": { "urls": 30, "requests": 222},
        "remaining": { "urls": 212, "requests": 1378},
    },
    "frontpage": {
        "matched": { "urls": 23, "requests": 181},
        "remaining": { "urls": 189, "requests": 1197},
    },
    "java": {
        "matched": { "urls": 6, "requests": 41},
        "remaining": { "urls": 183, "requests": 1156},
    },

    # new families in this iteration
    "lotus_domino": {
        "matched": { "urls": 18, "requests": 128},
        "remaining": { "urls": 165, "requests": 1028},
    },
    "asf": {
        "matched": { "urls": 72, "requests": 529},
        "remaining": { "urls": 93, "requests": 499},
    },
    "portal_app": {
        "matched": { "urls": 1, "requests": 4},
        "remaining": { "urls": 92, "requests": 495},
    },
    "file": {
        "matched": { "urls": 1, "requests": 4},
        "remaining": { "urls": 91, "requests": 491},
    }
})

verify_results(actual_result, expected_result)

Sample next batch of URLs for analysis from the filtered dataset

In [21]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/iisstart.asp,4,0
http://localhost:8080/iisstart.htm,9,0
http://localhost:8080/jstl-war/index.html,9,0
http://localhost:8080/mail.box,6,0
http://localhost:8080/msadc/msadcs.dll,16,0
http://localhost:8080/msadc/readme.txt,2,0
http://localhost:8080/msadc/samples/adctest.asp,6,0
http://localhost:8080/msadc/samples/selector/showcode.asp,8,0
http://localhost:8080/postinfo.html,5,0


**Observations:**  

Multiple Malformed URLs variations: 
- More variations of IIS scan probes - URLs with (`iisstart...`, `msadc...`, `samples...`)
- More variations Java metadata probes - URLs with `jstl-war...`
- More Lotus Domino databases probes - - URLs with (`mail.box`)
- More Microsoft frontpage scan probes - URLs with (`postinfo...`)  

In [22]:
# Define the patterns
more_iis_patterns = (r"iisstart|msadc")
some_more_iis_patterns = (r"/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/")
more_java_patterns = (r"jstl-war")
more_lotus_domino_patterns = (r"/mail\.box$")
more_frontpage_patterns = (r"postinfo\.html")

# Add more IIS family patterns
extend_pattern("iis", more_iis_patterns)
extend_pattern("iis", some_more_iis_patterns)

# Add more Java family patterns
extend_pattern("java", more_java_patterns)

# Add more  domino family patterns
extend_pattern("lotus_domino", more_lotus_domino_patterns)

# Add more frontpage family patterns
extend_pattern("frontpage", more_frontpage_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}/$
trailing_slash_suffix URLs: 1 URLs, 51 requests
remaining URLs:   280 URLs, 1885 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   260 URLs, 1749 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore
websphere_suffix URLs: 18 URLs, 149 requests
remaining URLs:   242 URLs, 1600 requests
-------------------------------

In [23]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (60, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    "e2ePortalProject"
  ],
  "file": [
    "databasenotes\\.html"
  ]
}


---

Sample next batch of URLs for analysis from the filtered dataset

In [24]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/scripts/tools/drvrs.exe,14,0
http://localhost:8080/scripts/tools/dsnform.exe,4,0
http://localhost:8080/scripts/tools/mkilog.exe,9,0
http://localhost:8080/scripts/tools/newdsn.exe,7,0
http://localhost:8080/servlet/ErrorReporter,5,0
http://localhost:8080/servlet/HelloWorldServlet,8,0
http://localhost:8080/servlet/SimpleServlet,8,0
http://localhost:8080/servlet/SnoopServlet,9,0
http://localhost:8080/servlet/aphtpassword,5,0


**Observations:**  

Multiple Malformed URLs variations: 
- More variations of IIS scan probes - URLs with (`/scripts/tools...`)
- More variations of Java metadata probes - URLs with `servlet...`)
- More variations of websphere probes - URLs with `techniques/servlets...`)
- Wordlist (generic bait) probe - URLs with (`theme`)
- Editor backup file suffix `~` *(emacs/vi convention)* - URLs with (`tienda1/...~`)
- URL ends with trailing slash - change earlier defined `trailing_slash_pattern` `(r"^https?://[^/:]+:\d{1,5}/$")` to cover for URLs api/endpoint/paths/ that ends with `/`

In [25]:
# Define the patterns
more_iis_patterns = (r"/scripts/tools/.*\.exe")
more_java_patterns = (r"/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)")
more_websphere_patterns = (r"techniques/servlets")
#known_scanner_bait_pattern = (r"^http://localhost:8080/theme$")
known_scanner_bait_pattern = (r"^https?://[^/:]+:\d{1,5}/theme$")
tilde_backup_file_pattern = (r"~$|~/|/~$")
overwrite_trailing_slash_pattern = (r"^https?://[^/:]+:\d{1,5}.*/$")

# Add more IIS family patterns
extend_pattern("iis", more_iis_patterns)

# Add more Java family patterns
extend_pattern("java", more_java_patterns)

# Add more websphere family patterns
extend_pattern("websphere", more_websphere_patterns)

# Add new scanner bait family and add patterns
new_pattern_family("known_scanner_bait")
extend_pattern("known_scanner_bait", known_scanner_bait_pattern)

# Add new tilde family and add patterns
new_pattern_family("tilde_backup_file")
extend_pattern("tilde_backup_file", tilde_backup_file_pattern)

# Overwrite trailing slash family and add patterns
overwrite_pattern("trailing_slash", overwrite_trailing_slash_pattern)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}.*/$
trailing_slash_suffix URLs: 41 URLs, 402 requests
remaining URLs:   240 URLs, 1534 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   220 URLs, 1398 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets
websphere_suffix URLs: 18 URLs, 140 requests
remaining URLs:   202 URLs, 1258 requests
-------

In [26]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (3, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}.*/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore",
    "techniques/servlets"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/",
    "/scripts/tools/.*\\.exe"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    "e2ePortalProject"
  ],
  "file": [
    "databasenotes\\.html"
  ],
  "known_scanner_bait": [
    "^https?:/

---

Sample next batch of URLs for analysis from the filtered dataset

In [27]:
anom_only_remaining_df.head(30)

label,anom,norm
url,,
http://localhost:8080/wliconsole,5,0
http://localhost:8080/worldmusic/action/catalog,5,0
http://localhost:8080/worldmusic/action/cdlist,4,0


**Observations:**  

Last few URL variations: 
- More variations of WebSphere scan probes - URLs with (`wliconsole`)
- More variations of Java probes - URLs with `worldmusic...`)

In [28]:
# Define the patterns
more_java_patterns = (r"worldmusic")
more_websphere_patterns = (r"wliconsole")

# Add more Java family patterns
extend_pattern("java", more_java_patterns)

# Add more websphere family patterns
extend_pattern("websphere", more_websphere_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java|jsp)
extensions_suffix URLs: 1308 URLs, 3398 requests
remaining URLs:   307 URLs, 2110 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 26 URLs, 174 requests
remaining URLs:   281 URLs, 1936 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}.*/$
trailing_slash_suffix URLs: 41 URLs, 402 requests
remaining URLs:   240 URLs, 1534 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   220 URLs, 1398 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets|wliconsole
websphere_suffix URLs: 19 URLs, 145 requests
remaining URLs:   201 URLs, 1253 reque

In [29]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (0, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java|jsp)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}.*/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore",
    "techniques/servlets",
    "wliconsole"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/",
    "/scripts/tools/.*\\.exe"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)",
    "worldmusic"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    "e2ePortalProject"
  ],
  "file": [
    "databasenotes\\.html"
  ],
  "k

The anomalous URLs are now fully bucketed into families. These can be used for feature engineering

---

In [30]:
# partition audit
assert len(anom_only_remaining_df) == 0

In [31]:
# partition audit
len(anom_only_remaining_df)

0

Verify if any `anom` URL family pattern match any `norm` URLs. This check is the evidence proof that makes these features usable in the model training

- Load the normal URL data set
- Look up which `anom` URL patterns hit the `norm` URLs
- Analyse the pattern and URLs from both classes
- Refine the patttern such that they hit no normal class URLs

procedure:  
Pull out the matching normal and anomalous URLs sets  
modify the pattern  
**make sure it hits no normal URLs but continue to hit the same set of anomalous URLs**

In [33]:
norm_only_urls_df.head(20)

label,anom,norm
url,,
http://localhost:8080/tienda1/global/creditos.jsp,0,1000
http://localhost:8080/tienda1/global/estilos.css,0,1000
http://localhost:8080/tienda1/global/menum.jsp,0,1000
http://localhost:8080/tienda1/global/titulo.jsp,0,1000
http://localhost:8080/tienda1/imagenes/1.gif,0,1000
http://localhost:8080/tienda1/imagenes/2.gif,0,1000
http://localhost:8080/tienda1/imagenes/3.gif,0,1000
http://localhost:8080/tienda1/imagenes/cmenbul.gif,0,1000
http://localhost:8080/tienda1/imagenes/logo.gif,0,1000


Use the derived pattern families expression to check for any overlap with normal URLs

In [92]:
# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
#norm_only_remaining_df = norm_only_urls_df
#print(norm_only_remaining_df.shape)

# Filter URLs on compiled patterns
#norm_only_remaining_df = filter_matched_urls(norm_only_remaining_df, compiled)

# Test compiled patterns on `norm_only` dataset. To avoid false positives, the `anom` compiled patterns should not match `norm` URL
norm_matched = filter_unmatched_urls(norm_only_urls_df, compiled)
#print(type(matched))
#for key, value in matched:
#    print(matched[key]['url'])

'extensions' family matched URLs: 10


**1. Pull out normal URLs matching extension pattern**

The `extensions` family pattern hits 10 `norm` URLs. Analyse the URLs

In [94]:
norm_matched['extensions'].head(10)

label,anom,norm
url,,
http://localhost:8080/tienda1/global/creditos.jsp,0,1000
http://localhost:8080/tienda1/global/menum.jsp,0,1000
http://localhost:8080/tienda1/global/titulo.jsp,0,1000
http://localhost:8080/tienda1/index.jsp,0,1000
http://localhost:8080/tienda1/miembros/fotos.jsp,0,1000
http://localhost:8080/tienda1/miembros/index.jsp,0,1000
http://localhost:8080/tienda1/miembros/salir.jsp,0,1000
http://localhost:8080/tienda1/publico/carrito.jsp,0,1000
http://localhost:8080/tienda1/publico/miembros.jsp,0,1000


**Observations:**  
The URLs begin with `http://localhost:8080/tienda1`

In [95]:
orig_norm_url_extension_dataset = norm_matched

Pull out the `extensions` pattern for reference

In [103]:
orig_extension_pattern = pattern_families['extensions'][0]
orig_extension_pattern

'\\.(?:bak|old|inc|java|jsp)'

**2. Pull out anomalous URLs matching extension pattern**

For URL comparison, pull Out the URLs from the `anom_only` data set that match extensions pattern
1. Run current extension pattern on `anom_only` dataset and get orig_anom_url_extension_dataset
2. Modify the extensions pattern => new_extensions_pattern
3. Run new_extensions_pattern on `norm_only` dataset => new_norm_extension_url_dataset
4. if new_extensions_pattern does not hit any `norm_only` dataset URLs (i.e len(new_norm_extension_url_dataset) == 0) then proceed to next step else go back to step 2
5. Run new_extensions_pattern on `anom_only` dataset => new_anom_extension_url_dataset
6. if new_anom_extension_url_dataset contains exactly the same URLs in the orig_anom_url_extension_dataset then proceed to next step else go back to step 2
7. Found the final extension pattern

1. Run current extension pattern on `anom_only` dataset and get orig_anom_url_extension_dataset

In [63]:
# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

#extensions_only_compiled = compiled_pattern(pattern_families['extensions'][0])
orig_anom_url_extension_dataset = filter_unmatched_urls(anom_only_remaining_df, {'extensions': pattern_families['extensions'][0]})

(1615, 2)
'extensions' family matched URLs: 1308


In [81]:
type(orig_anom_url_extension_dataset)
orig_anom_url_extension_dataset['extensions']

label,anom,norm
url,,
http://localhost:8080.BAK,19,0
http://localhost:8080.Bak,20,0
http://localhost:8080.INC,13,0
http://localhost:8080.Inc,25,0
http://localhost:8080.OLD,20,0
...,...,...
http://localhost:8080/tienda1/publico/vaciar.jsp~,2,0
http://localhost:8080/travelnet/home.jsp,11,0
http://localhost:8080/webapp/examples/simple.jsp,5,0


In [82]:
orig_anom_url_extension_dataset['extensions'].head(10)

label,anom,norm
url,,
http://localhost:8080.BAK,19,0
http://localhost:8080.Bak,20,0
http://localhost:8080.INC,13,0
http://localhost:8080.Inc,25,0
http://localhost:8080.OLD,20,0
http://localhost:8080.Old,22,0
http://localhost:8080.bak,23,0
http://localhost:8080.inc,16,0
http://localhost:8080.java,16,0


In [87]:
#orig_anom_url_extension_dataset['extensions'].str.contains(r"BAK", case=False, regex=True, na=False)
type(orig_anom_url_extension_dataset['extensions'])

pandas.core.frame.DataFrame

In [88]:
orig_anom_url_extension_dataset['extensions']

label,anom,norm
url,,
http://localhost:8080.BAK,19,0
http://localhost:8080.Bak,20,0
http://localhost:8080.INC,13,0
http://localhost:8080.Inc,25,0
http://localhost:8080.OLD,20,0
...,...,...
http://localhost:8080/tienda1/publico/vaciar.jsp~,2,0
http://localhost:8080/travelnet/home.jsp,11,0
http://localhost:8080/webapp/examples/simple.jsp,5,0


In [96]:
urls_a = orig_anom_url_extension_dataset['extensions'].index.to_series()
urls_n = orig_norm_url_extension_dataset['extensions'].index.to_series()

In [102]:
urls_a.str.contains((r"\.jsp"), case=False, regex=True, na=False).value_counts()

url
True     775
False    533
Name: count, dtype: int64

In [100]:
urls_n.str.contains((r"\.jsp"), case=False, regex=True, na=False).value_counts()

url
True    10
Name: count, dtype: int64

In [106]:
orig_extension_pattern

'\\.(?:bak|old|inc|java|jsp)'

In [105]:
urls_a.str.contains(orig_extension_pattern, case=False, regex=True, na=False).value_counts()

url
True    1308
Name: count, dtype: int64

In [104]:
urls_n.str.contains(orig_extension_pattern, case=False, regex=True, na=False).value_counts()

url
True    10
Name: count, dtype: int64

In [127]:
# modify the pattern
new_extension_pattern = '\\.(?:bak|old|inc|java)'
#print(urls_n.str.contains(new_extension_pattern, case=False, regex=True, na=False).value_counts())
mask = urls_n.str.contains(new_extension_pattern, case=False, regex=True, na=False)
print(f"URLs matched: {mask.sum()}")

url
False    10
Name: count, dtype: int64
URLs matched: 0


In [128]:
#urls_n.values

In [129]:
mask = urls_a.str.contains(new_extension_pattern, case=False, regex=True, na=False).value_counts()
print(f"URLs matched: {mask.sum()}")

URLs matched: 1308


Extract the `anom` that matched the `.jsp` extension pattern

In [86]:
# find the urls that match the .jsp pattern 
#anom_only_extensions_urls = anom_matched['extensions'].index.to_series()
#anom_only_extensions_urls[anom_only_extensions_urls['url'] == [anom_only_extensions_urls.str.contains(r"\.jsp")]]
#anom_only_extensions_urls.to_frame()['url'].head(10)

2. Modify the extensions pattern => new_extensions_pattern

In [132]:
# Define the patterns
# remove .jsp from original extension_pattern= r"\.(?:bak|old|inc|java|jsp)" 
overwrite_extensions_patterns = (r"\.(?:bak|old|inc|java)")

# Overwrite extensions family patterns
overwrite_pattern("extensions", overwrite_extensions_patterns)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java)
extensions_suffix URLs: 1117 URLs, 2573 requests
remaining URLs:   498 URLs, 2935 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 99 URLs, 542 requests
remaining URLs:   399 URLs, 2393 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}.*/$
trailing_slash_suffix URLs: 75 URLs, 541 requests
remaining URLs:   324 URLs, 1852 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   304 URLs, 1716 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets|wliconsole
websphere_suffix URLs: 25 URLs, 199 requests
remaining URLs:   279 URLs, 1517 requests


In [133]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (7, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}.*/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore",
    "techniques/servlets",
    "wliconsole"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/",
    "/scripts/tools/.*\\.exe"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)",
    "worldmusic"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    "e2ePortalProject"
  ],
  "file": [
    "databasenotes\\.html"
  ],
  "known

In [134]:
anom_only_remaining_df.head(10)

label,anom,norm
url,,
http://localhost:8080/.jsp,10,0
http://localhost:8080/admin/logon.jsp,10,0
http://localhost:8080/admin/secure/logon.jsp,5,0
http://localhost:8080/compass/logon.jsp,10,0
http://localhost:8080/portalAppAdmin/login.jsp,4,0
http://localhost:8080/travelnet/home.jsp,11,0
http://localhost:8080/ws-client/loanCalculation.jsp,7,0


In [136]:
# WebSphere admin console
extend_pattern("websphere", r"admin/(?:secure/)?logon\.jsp$")

# WebLogic console + portal admin (folds into the existing weblogic family)
extend_pattern("portal_app", r"compass/logon\.jsp$|portalAppAdmin/login\.jsp$")

# sample-app probes
extend_pattern("known_scanner_bait", r"travelnet/|ws-client/loanCalculation\.jsp$")

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java)
extensions_suffix URLs: 1117 URLs, 2573 requests
remaining URLs:   498 URLs, 2935 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 99 URLs, 542 requests
remaining URLs:   399 URLs, 2393 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}.*/$
trailing_slash_suffix URLs: 75 URLs, 541 requests
remaining URLs:   324 URLs, 1852 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   304 URLs, 1716 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets|wliconsole|admin/(?:secure/)?logon\.jsp$|admin/(?:secure/)?logon\.jsp$
websphere_suffix URLs: 27 U

In [139]:
# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Test compiled patterns on `norm_only` dataset. To avoid false positives, the `anom` compiled patterns should not match `norm` URL
norm_matched = filter_unmatched_urls(norm_only_urls_df, compiled)
len(norm_matched)

0

In [ ]:
the modifed pattern did not match any normal URLs. This is good.

We now have 1 unmatched anom to deal with

In [145]:
anom_only_remaining_df.head()

label,anom,norm
url,,
http://localhost:8080/.jsp,10,0


In [142]:
#print(len(anom_only_remaining_df))
#print(len(norm_matched))

In [157]:
misc_probes_pattern = (r"^https?://[^/:]+:\d{1,5}/\.jsp$")
# works: misc_probes_pattern = (r"http://.*:\d{1,5}/\.jsp$")
# sample-app probes
new_pattern_family("misc_probes")
extend_pattern("misc_probes", misc_probes_pattern)

# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Reset to original Dataset
anom_only_remaining_df = anom_only_urls_df
print(anom_only_remaining_df.shape)

# Filter URLs on compiled patterns
anom_only_remaining_df = filter_matched_urls(anom_only_remaining_df, compiled)

(1615, 2)
extensions: \.(?:bak|old|inc|java)
extensions_suffix URLs: 1117 URLs, 2573 requests
remaining URLs:   498 URLs, 2935 requests
--------------------------------------------------------------------------------
numeric-id: \d{10,}
numeric-id_suffix URLs: 99 URLs, 542 requests
remaining URLs:   399 URLs, 2393 requests
--------------------------------------------------------------------------------
trailing_slash: ^https?://[^/:]+:\d{1,5}.*/$
trailing_slash_suffix URLs: 75 URLs, 541 requests
remaining URLs:   324 URLs, 1852 requests
--------------------------------------------------------------------------------
adobe_coldfusion: CFIDE/
adobe_coldfusion_suffix URLs: 20 URLs, 136 requests
remaining URLs:   304 URLs, 1716 requests
--------------------------------------------------------------------------------
websphere: IBMWebAS|webapp|WebSphere|SmarTicketApp|estore|techniques/servlets|wliconsole|admin/(?:secure/)?logon\.jsp$|admin/(?:secure/)?logon\.jsp$
websphere_suffix URLs: 27 U

In [159]:
print(f"Dataframe shape: {anom_only_remaining_df.shape}")
print(f"pattern_families:\n{json.dumps(pattern_families, indent=2)}")

Dataframe shape: (0, 2)
pattern_families:
{
  "extensions": [
    "\\.(?:bak|old|inc|java)"
  ],
  "numeric-id": [
    "\\d{10,}"
  ],
  "trailing_slash": [
    "^https?://[^/:]+:\\d{1,5}.*/$"
  ],
  "adobe_coldfusion": [
    "CFIDE/"
  ],
  "websphere": [
    "IBMWebAS",
    "webapp|WebSphere|SmarTicketApp",
    "estore",
    "techniques/servlets",
    "wliconsole",
    "admin/(?:secure/)?logon\\.jsp$",
    "admin/(?:secure/)?logon\\.jsp$"
  ],
  "iis": [
    "IISSamples",
    "iisadmin|iisadmpwd",
    "iisstart|msadc",
    "/samples/(?:dbsamp|gbook|htmlsamp|isapi|sampsite)/",
    "/scripts/tools/.*\\.exe"
  ],
  "frontpage": [
    "_cti|_private|_vti",
    "postinfo\\.html"
  ],
  "java": [
    "WEB-INF",
    "\\.do$|flash/java/javabean|FlashJavaBean",
    "jstl-war",
    "/servlet/(?:Snoop|Hello|Simple|snoop2?|ErrorReporter|aphtpassword|hello)",
    "worldmusic"
  ],
  "lotus_domino": [
    "\\.nsf$",
    "/mail\\.box$"
  ],
  "asf": [
    "asf-logo-wide"
  ],
  "portal_app": [
    

In [158]:
# Compile the pattern
compiled = compiled_pattern(pattern_families)

# Test compiled patterns on `norm_only` dataset. To avoid false positives, the `anom` compiled patterns should not match `norm` URL
norm_matched = filter_unmatched_urls(norm_only_urls_df, compiled)
len(norm_matched)

0